In [2]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs=["text"],
    api_name="predict"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [4]:
import gradio as grb
from IPython.display import display, Markdown, update_display 

In [5]:
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

openai_api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()

In [6]:
models = groq.models.list()
for model in models.data:
    print(f"Model ID: {model.id}")
    # display(Markdown(f"**Object Type:** {model.object}"))
    # display(Markdown(f"**Created At:** {model.created}"))
    # display(Markdown(f"**Owned By:** {model.owned_by}"))
    # display(Markdown("---"))

Model ID: openai/gpt-oss-safeguard-20b
Model ID: whisper-large-v3-turbo
Model ID: allam-2-7b
Model ID: meta-llama/llama-prompt-guard-2-22m
Model ID: meta-llama/llama-prompt-guard-2-86m
Model ID: qwen/qwen3.8-27b
Model ID: canopylabs/orpheus-v1-english
Model ID: openai/gpt-oss-20b
Model ID: openai/gpt-oss-120b
Model ID: canopylabs/orpheus-arabic-saudi
Model ID: whisper-large-v3


In [7]:
model_groq_gpt_oss_120b = "openai/gpt-oss-120b"

In [8]:
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=messages)
    return response.choices[0].message.content

In [9]:
message_gpt("What is today's date?")

'Today is November 9, 2026.'

## UI Layer

In [10]:
def shout(text):
    print("Shout has been called with text:", text)
    return text.upper()

In [11]:
app = gr.Interface(
    fn = shout,
    inputs = "textbox",
    outputs = "textbox",
    flagging_mode="never")

app.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Let's make a Gpt Chatbot Basic

In [12]:
messsage_input = gr.Textbox(label="Enter your message", placeholder="Type your message here...")
message_output = gr.Textbox(label="Response", placeholder="The response will appear here...", interactive=False)

chatBot = gr.Interface(
    fn=message_gpt,
    title="Groq GPT Chatbot",
    inputs=[messsage_input],
    outputs=[message_output],
    examples=[
        ["What is the weather like today?"],
        ["Can you tell me a joke?"],
        ["What is the capital of France?"]
    ],
    allow_flagging="never"
)

chatBot.launch()

c:\Users\saree\Desktop\AI ENG\AI-Engineering\.venv\Lib\site-packages\gradio\interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [13]:
# Let's use Markdown
# Are you wondering why it makes any difference to set system_message when it's not referred to in the code below it?
# I'm taking advantage of system_message being a global variable, used back in the message_gpt function (go take a look)
# Not a great software engineering practice, but quite common during Jupyter Lab R&D!

system_message = "You are a helpful assistant that responds in markdown without code blocks"
# Global Variable Declared Earlier

message_input = gr.Textbox(label="Your message:", info="Enter a message for openai/gpt-oss-120b", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_gpt,
    title="GPT", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Let's go with Streaming Like ChatGPT

In [15]:
def stream_message_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    chunks = openai.chat.completions.create(
        model="gpt-6-luna",
        messages=messages,
        stream=True,
        reasoning_effort="none"
    )
    
    result = ""
    for chunk in chunks:
        result += chunk.choices[0].delta.content or ""
        yield result

In [17]:
message_input = gr.Textbox(label="Hi, How may I help you today?", lines=7)
message_output = gr.Textbox(label="Response:", lines=7)

app = gr.Interface(
    fn=stream_message_gpt,
    inputs=[message_input],
    outputs=[message_output],
    title="Streaming ChatGPT-like Interface",
    description="This interface streams responses from the GPT model in real-time.",
    allow_flagging="never",
    examples=[
        ["What is the weather like today?"],
        ["Can you tell me a joke?"],
        ["What is the capital of France?"]
    ]
)

app.launch()

c:\Users\saree\Desktop\AI ENG\AI-Engineering\.venv\Lib\site-packages\gradio\interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


##